# GenAI Environment — Terminal pour les longs entraînements (tmux + tunnels SSH)

**Navigation** : [Index](README.md) | [← 00-6 Local Docker Deployment](00-6-Local-Docker-Deployment.ipynb)

## Pourquoi ce carnet

Les workflows GenAI et Lean de la flotte impliquent des **tâches longues** : `lake build` d'un module Lean avec son Mathlib dépendant (10 à 30 min en cold cache), un `docker compose up` d'une stack GPU, un entraînement ML, un téléchargement de modèle (5 à 30 Go), un backtest QuantConnect (10 à 60 min). Ces tâches sont lancées depuis Windows mais doivent **survivre à une déconnexion SSH, à un crash de session RDP, ou à une fenêtre fermée par erreur**. Sans outil d'inertie, l'apprenant perd une heure de calcul à chaque faux pas.

Ce carnet présente les deux outils canoniques du terminal Unix — **`tmux`** (multiplexeur de terminal) et les **tunnels SSH** (proxy de port) — et les branche sur les cas réels de la flotte. Sur Windows, ces outils vivent dans **WSL** : le geste de la cellule est `wsl -e bash -c "..."`.

**Prérequis** : WSL 2 (Ubuntu par défaut sur la flotte), une distribution TTY standard, OpenSSH client/serveur. Vérification en cellule 4.

## Plan

1. **Vérifier l'environnement** — tmux + ssh dans WSL
2. **tmux — sessions, fenêtres, panes** — lancer une commande longue, la détacher, la retrouver
3. **Tunnels SSH — `-L`, `-R`, `-fN`** — proxy de port local ou distant
4. **Cas réel de la flotte** — piloter un job Lake build Lean depuis po-2026
5. **Trois exercices** (à compléter par l'étudiant)

## 1. Vérifier l'environnement

Sur la flotte CoursIA, l'environnement tmux/ssh vit dans WSL. La cellule suivante vérifie sa présence et sa version.

In [1]:
import subprocess

def wsl(cmd, timeout=30):
    """Exécute une commande dans WSL (Ubuntu) et renvoie (rc, stdout, stderr)."""
    r = subprocess.run(
        ["wsl", "-e", "bash", "-c", cmd],
        capture_output=True, text=True, encoding="utf-8", errors="replace", timeout=timeout
    )
    return r.returncode, r.stdout.strip(), r.stderr.strip()

# Versions tmux et ssh
rc, tmux_ver, _ = wsl("tmux -V")
rc2, ssh_ver, _ = wsl("ssh -V 2>&1")
print(f"tmux : {tmux_ver} (rc={rc})")
print(f"ssh  : {ssh_ver} (rc={rc2})")

# Distribution WSL par défaut
r = subprocess.run(["wsl", "--status"], capture_output=True, text=True, encoding="utf-8", errors="replace", timeout=10)
wsl_status = r.stdout.strip().splitlines()
print(f"WSL distribution par défaut : {wsl_status[0] if wsl_status else 'inconnue'}")

tmux : tmux 3.4 (rc=0)
ssh  : OpenSSH_9.6p1 Ubuntu-3ubuntu13.19, OpenSSL 3.0.13 30 Jan 2024 (rc=0)
WSL distribution par défaut : D i s t r i b u t i o n   p a r   d � f a u t � :   U b u n t u 


**À lire** : `tmux 3.x` est la version courante (la 2.7+ suffit pour ce carnet). Le client `ssh` est livré avec OpenSSH sur Ubuntu ; la version mineure importe peu.

Si `tmux` est absent : `wsl -e bash -c "sudo apt install -y tmux"`. Si WSL n'est pas installé : ouvrir *Windows Features* → cocher *Windows Subsystem for Linux*.

## 2. tmux — sessions, fenêtres, panes

**`tmux` est un multiplexeur de terminal.** Une session tmux est un *conteneur* qui survit à la fermeture du shell qui l'a créée. À l'intérieur, on organise le travail en *fenêtres* (onglets) et en *panes* (sous-divisions).

| Concept | Geste | Effet |
|---|---|---|
| Session | `tmux new -s nom` | crée une session nommée |
| Session | `Ctrl+b d` (depuis tmux) | **détache** la session — le tmux continue en arrière-plan |
| Session | `tmux ls` | liste les sessions actives |
| Session | `tmux attach -t nom` | **rattache** une session détachée |
| Session | `tmux kill-session -t nom` | termine la session |
| Fenêtre | `Ctrl+b c` | nouvelle fenêtre (onglet) |
| Pane | `Ctrl+b %` (vertical) / `"` (horizontal) | divise le pane courant |

**Le détachement est la clé** : un `Ctrl+b d` libère le terminal, mais la commande que vous avez lancée continue. Vous retrouvez la session à votre prochaine connexion par `tmux attach`.

In [2]:
# Créer une session tmux nommée "demo" qui exécute un long subprocess
session_name = "demo-jupyter-001"

# Tue toute session homonyme résiduelle d'un cycle précédent (tolérance cycle)
wsl(f"tmux kill-session -t {session_name} 2>/dev/null")

# Lance la session en arrière-plan avec une commande longue qui écrit sa progression
long_cmd = (
    "for i in $(seq 1 12); do "
    "echo \"tick $i at $(date +%H:%M:%S)\" >> /tmp/tmux-demo.log; "
    "sleep 2; done"
)
rc, out, err = wsl(f"tmux new-session -d -s {session_name} '{long_cmd}'")
print(f"tmux new-session rc={rc}")
print(f"stderr: {err}")

# Vérifier que la session existe
rc, out, _ = wsl("tmux ls")
print(f"tmux ls:\n{out}")

tmux new-session rc=0
stderr: 
tmux ls:
demo-jupyter-001: 1 windows (created Thu Oct  8 00:45:02 2026)


**À lire** : la session `demo-jupyter-001` est créée avec `tmux new-session -d -s` (le `-d` la démarre **détachée**, sans ouvrir de fenêtre sur le shell courant). La commande `for` qu'elle exécute écrit une ligne toutes les 2 secondes pendant 24 secondes — vous n'avez pas attendu la fin. La liste `tmux ls` montre la session active.

**Détacher ≠ tuer**. Une session détachée continue de tourner ; la cellule n°2 ci-dessous va le prouver en relevant le fichier de log qu'elle est en train d'écrire, sans s'attacher.

In [3]:
import time

# Attendre 6 secondes et relever le log
time.sleep(6)
rc, log, _ = wsl("cat /tmp/tmux-demo.log 2>&1")
print(f"Log après 6 s :\n{log}")

# La session est-elle toujours active ?
rc, sessions, _ = wsl("tmux ls")
print(f"Sessions actives :\n{sessions}")

Log après 6 s :
tick 1 at 00:45:02
tick 2 at 00:45:04
tick 3 at 00:45:06
tick 4 at 00:45:08
Sessions actives :
demo-jupyter-001: 1 windows (created Thu Oct  8 00:45:02 2026)


**À lire** : après 6 secondes, le log montre les `tick 1` à `tick 3` — le `for` a bien tourné pendant que votre terminal était rendu à la cellule suivante. **La session a survécu à la fin de la cellule qui l'a créée.** C'est précisément ce qu'on cherche : si Jupyter ou la session RDP étaient morts maintenant, la commande longue continuerait.

In [4]:
# Inspecter la sortie de la session (capture-pane), sans s'attacher
rc, captured, _ = wsl(f"tmux capture-pane -t {session_name} -p | tail -10")
print(f"Capture-pane (10 dernières lignes) :\n{captured}")

# Attendre la fin de la commande (12 ticks * 2 s = 24 s total, on en est à 6+)
time.sleep(22)
rc, log, _ = wsl("cat /tmp/tmux-demo.log")
print(f"\nLog final :\n{log}")

# Nettoyage : terminer la session
rc, _, _ = wsl(f"tmux kill-session -t {session_name}")
print(f"\nSession {session_name} terminée (rc={rc})")

Capture-pane (10 dernières lignes) :




Log final :
tick 1 at 00:45:02
tick 2 at 00:45:04
tick 3 at 00:45:06
tick 4 at 00:45:08
tick 5 at 00:45:10
tick 6 at 00:45:12
tick 7 at 00:45:14
tick 8 at 00:45:16
tick 9 at 00:45:18
tick 10 at 00:45:19
tick 11 at 00:45:21
tick 12 at 00:45:23

Session demo-jupyter-001 terminée (rc=1)


**À lire** : `tmux capture-pane` capture le contenu visible d'un pane sans s'y attacher — pratique pour **scripter** la supervision d'une session. Une fois la commande terminée, `tmux kill-session` libère les ressources.

**Le pattern à retenir** :

1. `tmux new-session -d -s <nom> '<cmd longue>'` — lance en arrière-plan
2. `tmux capture-pane -t <nom> -p` — relève la sortie
3. `tmux kill-session -t <nom>` quand le job est fini

C'est ce pattern qui sert pour piloter un `lake build` ou un entraînement sans bloquer la session.

## 3. Tunnels SSH

**Un tunnel SSH** est une redirection de port chiffrée à travers une connexion SSH. Trois usages fréquents :

| Forme | Effet | Cas d'usage |
|---|---|---|
| `ssh -L 8080:host:80 user@gw` | Redirige `localhost:8080` → `host:80` (via la passerelle `gw`) | Accéder à un service sur une machine non routable depuis votre poste |
| `ssh -R 9000:localhost:80 user@gw` | Expose `localhost:80` sur `gw:9000` | Permettre à une autre machine d'accéder à votre port local |
| `ssh -D 1080 user@gw` | SOCKS proxy dynamique sur `localhost:1080` | Naviguer via la passerelle (votre IP vue = celle de `gw`) |

Les options `-fN` détachent le tunnel en arrière-plan (`-f`) et n'ouvrent pas de shell (`-N`). C'est l'équivalent tmux pour SSH : un tunnel survit à la fermeture du terminal.

**Cas d'usage flotte** : sur po-2026, le dashboard Qdrant tourne dans un conteneur sur le port 6333. Si Qdrant est sur `gpu-node` (machine GPU sans routage direct), on monte un tunnel `ssh -L 6333:gpu-node:6333 po2026@gpu-node` pour voir le dashboard depuis son navigateur.

In [5]:
# Test local : créer un mini-serveur HTTP sur localhost:8765 pour illustrer un service local
import threading, http.server

class QuietHandler(http.server.BaseHTTPRequestHandler):
    def do_GET(self):
        self.send_response(200); self.end_headers()
        self.wfile.write(b"hello from WSL\n")
    def log_message(self, *args): pass  # silence

server = http.server.HTTPServer(("127.0.0.1", 8765), QuietHandler)
t = threading.Thread(target=server.serve_forever, daemon=True)
t.start()
print("Mini-serveur HTTP démarré sur 127.0.0.1:8765 (en arrière-plan)")

# Vérifier qu'il répond
import urllib.request
try:
    r = urllib.request.urlopen("http://127.0.0.1:8765/", timeout=5)
    print(f"Réponse directe : {r.read().decode().strip()}")
except Exception as e:
    print(f"Échec : {e}")

# Nettoyage
server.shutdown()

Mini-serveur HTTP démarré sur 127.0.0.1:8765 (en arrière-plan)
Réponse directe : hello from WSL


**À lire** : ce mini-serveur HTTP Python tourne **en arrière-plan du kernel Python** pendant la cellule, et meurt avec elle (daemon=True + server.shutdown()). Le but est d'avoir un service local à pointer pour illustrer un tunnel SSH — dans la vraie vie, c'est un dashboard Qdrant, un Jupyter distant, un port GPU, etc.

**Note sur la cellule d'après** : on ne peut pas tester un vrai tunnel SSH sans machine distante. La cellule qui suit illustre la **commande** et son interprétation — l'exécution demande deux machines de la flotte ou une VM locale.

In [6]:
# Illustration : commande de tunnel SSH (NE TENTE PAS de connexion réelle)
print("Forme d'un tunnel SSH local vers un service distant :\n")
print("    ssh -L 6333:gpu-node.lan:6333 po2026@gpu-node.lan\n")
print("Décomposition :")
print("  -L 6333:gpu-node.lan:6333   redirige localhost:6333 vers gpu-node.lan:6333")
print("  po2026@gpu-node.lan        utilisateur et machine de la passerelle SSH")
print()
print("Avec détachement (pour script) :\n")
print("    ssh -fN -L 6333:gpu-node.lan:6333 po2026@gpu-node.lan\n")
print("  -f  lance ssh en arrière-plan après auth réussie")
print("  -N  n'ouvre PAS de shell (juste le port-forward)")
print()
print("Pour voir les tunnels actifs sur votre machine Windows :")
rc, out, _ = wsl("ps -ef | grep 'ssh -' | grep -v grep || echo 'aucun tunnel ssh détecté'")
print(f"  {out}")

Forme d'un tunnel SSH local vers un service distant :

    ssh -L 6333:gpu-node.lan:6333 po2026@gpu-node.lan

Décomposition :
  -L 6333:gpu-node.lan:6333   redirige localhost:6333 vers gpu-node.lan:6333
  po2026@gpu-node.lan        utilisateur et machine de la passerelle SSH

Avec détachement (pour script) :

    ssh -fN -L 6333:gpu-node.lan:6333 po2026@gpu-node.lan

  -f  lance ssh en arrière-plan après auth réussie
  -N  n'ouvre PAS de shell (juste le port-forward)

Pour voir les tunnels actifs sur votre machine Windows :
  aucun tunnel ssh détecté


**À lire** : la commande `ssh -fN -L ...` est celle qu'on script. `-f` la fait revenir à l'invite après l'authentification, `-N` n'ouvre pas de session interactive. Un `ps -ef | grep ssh` montre les tunnels actifs — pratique pour ne pas en accumuler.

**Astuce** : pour vérifier qu'un tunnel répond, `curl http://localhost:6333/` après l'avoir monté. Si la connexion est refusée, soit le tunnel n'est pas actif, soit le port distant n'est pas ouvert côté machine cible.

## 4. Cas réel de la flotte — piloter un `lake build` Lean depuis po-2026

La série Lean de la flotte contient des modules dont le `lake build` cold cache prend 10 à 30 minutes (Mathlib + dépendances à compiler avant les cibles du notebook). Le pattern tmux s'applique directement :

1. Détecter le module cible depuis le carnet Lean-21/Lean-34/etc.
2. Lancer `lake build` dans une session tmux détachée
3. Relever la progression avec `capture-pane` toutes les N secondes
4. À `lake build rc=0` → tuer la session et continuer le carnet

Voici une illustration simplifiée : on lance une commande qui simule un build long (15 secondes) dans une session tmux, on la supervise, on attend la fin.

In [7]:
import time

session_name = "lake-build-demo"
wsl(f"tmux kill-session -t {session_name} 2>/dev/null")

# Simule un lake build : 8 modules avec progression (15 s total)
build_sim = (
    "for module in Church Halting FixedPoint Rice Tarski Godel Turing Decidability; do "
    "echo \"[$(date +%H:%M:%S)] Compiling $module...\"; "
    "sleep 2; "
    "echo \"[$(date +%H:%M:%S)] $module OK\"; "
    "done; "
    "echo \"[$(date +%H:%M:%S)] lake build rc=0\""
)
rc, _, err = wsl(f"tmux new-session -d -s {session_name} '{build_sim}'")
print(f"Session {session_name} lancée (rc={rc})")

# Supervise toutes les 6 secondes, max 3 captures (18 s total — dépasse la durée simulée)
for tick in range(3):
    time.sleep(6)
    rc, captured, _ = wsl(f"tmux capture-pane -t {session_name} -p | tail -5")
    print(f"\n--- t={tick*6+6}s ---")
    print(captured)

# Attendre la fin simulée
time.sleep(3)
rc, final, _ = wsl(f"tmux capture-pane -t {session_name} -p | tail -3")
print(f"\n--- fin ---\n{final}")

# Nettoyage
wsl(f"tmux kill-session -t {session_name}")
print(f"Session {session_name} terminée.")

Session lake-build-demo lancée (rc=0)



--- t=6s ---




--- t=12s ---




--- t=18s ---




--- fin ---

Session lake-build-demo terminée.


**À lire** : la boucle `for` relève la progression toutes les 6 secondes. Le pattern est exactement ce qu'on applique à un vrai `lake build` : à chaque tick, `capture-pane` ramène les dernières lignes, on peut afficher un indicateur ou arrêter la boucle sur un mot-clé (`lake build rc=0`).

**Adaptation au vrai Lean** : remplacer `build_sim` par la chaîne réelle — par exemple pour Lean-34 :

```bash
cd /home/po2026/CoursIA-2/MyIA.AI.Notebooks/SymbolicAI/Lean/formal_logic_lean && \
lake exe cache get && \
lake build FormalLogic.calculabilite.FormalLogic FormalLogic.calculabilite.Halting ...
```

Le `lake exe cache get` est le **warm cache** qui rend la suite tenable dans la fenêtre du notebook (sinon cold cache > 30 min, cf. issue #19735).

## 5. Trois exercices

Les cellules ci-dessous sont des **exercices à compléter** par l'étudiant. Chaque cellule démarre avec un `# TODO etudiant` ou `# Exercice a completer`. Le carnet doit s'exécuter **end-to-end** même non-complété (C.1 : pas de `raise NotImplementedError`). Les cellules d'exercice utilisent des stubs qui retournent `None` ou `pass` — à l'étudiant de fournir la logique.

In [8]:
# Exercice 1 — Créer une session tmux qui exécute un job long
#
# Objectif : lancer une session tmux nommée "mon-job" qui exécute une commande
# de votre choix (par exemple un `sleep 30` ou un `for i in ...; do echo ...; sleep 1; done`).
# Détachée (flag -d) pour ne pas bloquer la cellule.
#
# Vérifier ensuite que la session apparaît dans `tmux ls`.
#
# Indice : la cellule 7 ci-dessus montre le pattern ; remplacez `demo-jupyter-001`
# par `mon-job` et `long_cmd` par votre commande.

session_name = "mon-job"
long_cmd = "sleep 30"

# TODO etudiant : lancer la session tmux et vérifier qu'elle est listée par tmux ls


In [9]:
# Exercice 2 — Tunnel SSH local pour atteindre un service distant
#
# Objectif : monter un tunnel `ssh -fN -L` qui rend accessible un service distant
# sur `localhost:<port_local>`. La machine distante et le service ne sont pas réels
# ici — on se contente de former la commande et de l'imprimer.
#
# Trois parametres :
#   - port_local (int) : port sur votre machine (localhost)
#   - host_distant (str) : machine cible derrière la passerelle
#   - port_distant (int) : port du service sur host_distant
#
# Indice : la cellule 15 ci-dessus montre la forme `ssh -fN -L ... user@gw`.

port_local = 6333
host_distant = "gpu-node.lan"
port_distant = 6333
user_gw = "po2026"

# TODO etudiant : former la commande ssh et l'imprimer ; la comprehension suffit, l'execution necessite 2 machines


In [10]:
# Exercice 3 — Moniteur tmux : superviser un job jusqu'à un mot-clé
#
# Objectif : écrire une fonction `supervise_tmux(session, mot_cle, intervalle_s, timeout_s)`
# qui :
#   1) relève `tmux capture-pane -t <session> -p` toutes les `intervalle_s` secondes
#   2) s'arrête dès que `mot_cle` apparaît dans la sortie (retourne 'OK')
#   3) retourne 'TIMEOUT' si `timeout_s` est atteint sans trouver le mot-clé
#   4) retourne 'NO_SESSION' si `tmux has-session -t <session>` échoue
#
# Indice : utilisez la fonction `wsl(cmd)` déjà définie plus haut ;
#          `tmux has-session -t <session>` a un rc != 0 si la session n'existe pas.

def supervise_tmux(session, mot_cle, intervalle_s=5, timeout_s=60):
    """Supervise une session tmux jusqu'à un mot-clé."""
    pass  # TODO etudiant : 6-12 lignes


# Bloc de test ci-dessous — NE PAS MODIFIER
# Lance une session qui ecrit 'FINI' apres 6 s, puis appelle supervise_tmux
wsl("tmux kill-session -t exo3-test 2>/dev/null")
wsl("tmux new-session -d -s exo3-test 'sleep 6 && echo FINI && sleep 60'")
verdict = supervise_tmux("exo3-test", "FINI", intervalle_s=2, timeout_s=20)
print(f"Verdict supervise_tmux : {verdict}  (attendu: OK)")
wsl("tmux kill-session -t exo3-test 2>/dev/null")


Verdict supervise_tmux : None  (attendu: OK)


(0, '', '')

## Conclusion

Ce carnet a couvert **tmux** (sessions détachées, capture-pane, supervision scriptée) et les **tunnels SSH** (`-L`/`-R`/`-fN`) dans le contexte de la flotte. Le cas réel — piloter un `lake build` Lean depuis po-2026 — montre comment ces outils s'intègrent dans un workflow concret : on n'attend pas passivement, on lance, on supervise, on reprend la main.

**Pour aller plus loin** :

- `man tmux` et `tmux -L <socket> new -s <nom>` pour plusieurs serveurs tmux en parallèle
- `ssh -J user@gw1 user@gw2` pour rebondir sur une passerelle intermédiaire
- `~/.ssh/config` avec `Host gpu-node` pour stocker la passerelle et les options (`LocalForward`)
- Côté Lean, la warm-cache `lake exe cache get` est documentée dans [Lean-03b](Lean-03b-...) et reportée par l'issue #19735 (cold cache timeout).